In [ ]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-20b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply

print("Total for 6 cupcakes:", get_price("cupcake", 6))
while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

   [log] get_price('cupcake', 6) -> 480
Total for 6 cupcakes: 480
You: hello
Bot: Hi! 👋 Welcome to Sweet Crumbs Home Bakery. How can I help you today?
You: menu please
Bot: Sure! Here’s our menu (prices in ₹):

- Plum Cake – ₹500  
- Chocolate Cake – ₹650  
- Cupcake – ₹80  

All cakes have egg‑less versions. Let me know what you’d like!
You: how many plum cakes are available?
Bot: I’ll check with the owner. May I know your name?
You: Ok, then I Want To Buy Five Cupcake, 6 Chocolate Cake
Bot: Thanks for your order! I’ve noted 5 cupcakes and 6 chocolate cakes. Could I have your name to place the order? (All orders need 24‑hour notice.)
You: how much is the bill?
Bot: Your total comes to ₹4,300. Please note we need 24‑hour notice for orders. Let me know your name to finalize.
You: Can You Tell Total Calculations Of The Bill
Bot: Sure!  
5 cupcakes × ₹80 = ₹400  
6 chocolate cakes × ₹650 = ₹3,900  
**Total bill = ₹400 + ₹3,900 = ₹4,300**.
You: thank you.
Bot: You’re welcome! If you need a